# Portafolio Cuantitativo Neutral al Mercado

## Extensión: Remuestreo y costos reales por pata

**Autor:** Samuel

**Institución:** Tec de Monterrey

**Área:** Finanzas Cuantitativas, Economía Estadística y Machine Learning

**Notebooks previos:** `Backtest-Corregido.ipynb` y  `Beta-Anclado-OU.ipynb`

# Remuestreo y costos reales por pata

**Pregunta:** ¿cambia algo si miramos la canasta USDCAD/USDNOK/USDSEK/USDSGD cada 4 u 8 horas en vez de cada hora, y si usamos costos realistas por par en lugar de 1 pb uniforme?

**Por qué:** (1) menos ruido de cotización por barra, menos operaciones espurias; (2) el costo real difiere por divisa (USDNOK y USDSEK suelen costar más que USDCAD), y eso decide si el break-even de ~1 pb era realista.

**Plan:** Fase 1 datos y remuestreo → Fase 2 motor (reutilizado de Beta-Anclado-OU) → Fase 3 walk-forward por frecuencia → Fase 4 costos por pata → Fase 5 conclusión.

## Fase 1 – Datos y remuestreo
Descargar las 4 divisas por hora, renombrar por ticker con verificación de niveles, y construir las series de 4h y 8h tomando el último precio de cada barra. Verificar barras por día (~6 en 4h, ~3 en 8h) para detectar huecos.

In [ ]:
import numpy as np, pandas as pd, yfinance as yf

In [3]:
tickers = {'USDCAD':'USDCAD=X','USDNOK':'USDNOK=X','USDSEK':'USDSEK=X','USDSGD':'USDSGD=X'}
raw = yf.download(list(tickers.values()), period='729d', interval='60m', progress=False, auto_adjust=True)['Close']
ticker_to_name = {v: k for k, v in tickers.items()}
data_1h = raw.rename(columns=ticker_to_name)[list(tickers.keys())].ffill().dropna()
if data_1h.index.tz is not None:
    data_1h.index = data_1h.index.tz_convert('UTC').tz_localize(None)
print(data_1h.shape)

(17322, 4)


In [4]:
def resample_bars(df, rule):
    # último precio de cada barra; se quitan barras vacías (fines de semana)
    return df.resample(rule).last().dropna()

data_4h = resample_bars(data_1h, '4h')
data_8h = resample_bars(data_1h, '8h')
for name, d in {'1h':data_1h, '4h':data_4h, '8h':data_8h}.items():
    print(name, d.shape, 'barras/día ≈', round(len(d)/d.index.normalize().nunique(), 1))

1h (17322, 4) barras/día ≈ 21.2
4h (4455, 4) barras/día ≈ 5.4
8h (2273, 4) barras/día ≈ 2.8


## Fase 2 – Motor (reutilizado de Beta-Anclado-OU)
Parámetros y funciones: calibración de β con OLS, ajuste del OU, máquina de estados de posiciones y búsqueda de umbrales con costo. Se adapta para que reciba cualquier frecuencia y un costo por pata (no uno uniforme).

In [9]:
from scipy.signal import lfilter

In [7]:
Y  = 'USDCAD'
XS = ['USDNOK','USDSEK','USDSGD']
TEST_DAYS = 30
MIN_TRAIN = 240        # días; primer día de prueba
STOP      = 3.5
FIXED_KE  = (2.0, 0.5)
COST_BPS  = {'USDCAD':1.0, 'USDNOK':1.0, 'USDSEK':1.0, 'USDSGD':1.0}   # por pata; se reemplaza en la Fase 4

In [10]:
try:
    from numba import njit
except ImportError:
    njit = lambda f: f

def calibrate(train):
    A = np.column_stack([np.ones(len(train))] + [train[c].values for c in XS])
    coef, *_ = np.linalg.lstsq(A, train[Y].values, rcond=None)
    alpha, beta = coef[0], coef[1:]
    s = train[Y].values - train[XS].values @ beta - alpha
    return alpha, beta, s

def fit_ou(s):
    x, y = s[:-1], s[1:]
    b, a = np.polyfit(x, y, 1)
    if not (0.0 < b < 1.0):
        return None
    resid = y - (a + b*x)
    return dict(b=b, mu=a/(1-b), sig=resid.std(ddof=2)/np.sqrt(1-b*b), hl=np.log(2)/(-np.log(b)))

In [11]:
@njit
def positions_from_z(z, k, e, stop):
    n = len(z); pos = np.zeros(n); p = 0.0; blocked = 0.0
    for t in range(n):
        zt = z[t]
        if blocked != 0.0 and abs(zt) < k:
            blocked = 0.0
        if p == 0.0:
            if blocked == 0.0:
                if zt >= k: p = -1.0
                elif zt <= -k: p = 1.0
        else:
            if p < 0 and zt <= e: p = 0.0
            elif p > 0 and zt >= -e: p = 0.0
            elif abs(zt) >= stop:
                blocked = 1.0 if zt > 0 else -1.0
                p = 0.0
        pos[t] = p
    return pos

## Fase 3 – Walk-forward por frecuencia
Mismas ventanas de prueba (30 días) para 1h, 4h y 8h, con β anclado y costo de 1 pb como referencia. Se compara bruto, neto, operaciones y break-even. Pregunta: ¿cambia el bruto al reducir el ruido de cotización?

In [18]:
from scipy import stats

In [13]:
def leg_cost(df, beta, costs):
    # costo (en precio) de mover 1 unidad de posición: suma del nocional de cada pata x su costo
    c = df[Y].values * costs[Y] / 1e4
    for i, name in enumerate(XS):
        c = c + np.abs(beta[i]) * df[name].values * costs[name] / 1e4
    return c

def ou_band_search(b, c_side_z, n=60000, seed=0, ks=np.arange(0.75, 3.01, 0.25), es=(0.0, 0.25, 0.5, 1.0)):
    eps = np.random.default_rng(seed).standard_normal(n) * np.sqrt(1 - b*b)
    z = lfilter([1.0], [1.0, -b], eps); dz = np.diff(z); best = None
    for k in ks:
        for e in es:
            if e >= k: continue
            pos = positions_from_z(z, k, e, STOP)
            rate = ((pos[:-1]*dz).sum() - c_side_z*np.abs(np.diff(pos, prepend=0.0)).sum()) / n
            if best is None or rate > best[2]:
                best = (k, e, rate)
    return best if best and best[2] > 0 else None     # None = abstenerse

def run_cfg(df, train_days, mode, costs=COST_BPS, seed=0):
    day = df.index.normalize().values; ud = np.unique(day); rows = []
    for j in range(MIN_TRAIN, len(ud) - TEST_DAYS + 1, TEST_DAYS):
        tr = df[(day >= ud[j-train_days]) & (day < ud[j])]
        te = df[(day >= ud[j]) & (day < ud[j+TEST_DAYS])]
        row = dict(window=j, gross=0.0, net=0.0, cost=0.0, ops=0, k=np.nan, e=np.nan, hl=np.nan)
        alpha, beta, s = calibrate(tr); ou = fit_ou(s)
        if ou is None or len(te) < 20:
            rows.append(row); continue
        E = (tr[Y].values + tr[XS].values @ np.abs(beta)).mean()           # normalizador
        c_side_z = leg_cost(tr, beta, costs).mean() / ou['sig']
        row['hl'] = ou['hl']
        if mode == 'fixed':
            k, e = FIXED_KE
        else:
            res = ou_band_search(ou['b'], c_side_z, seed=seed)
            if res is None:
                rows.append(row); continue
            k, e = res[0], res[1]
        s_te = te[Y].values - te[XS].values @ beta - alpha                  # pesos congelados
        pos = positions_from_z((s_te - ou['mu']) / ou['sig'], k, e, STOP)
        gross = (pos[:-1] * np.diff(s_te)).sum()
        cost = (np.abs(np.diff(pos, prepend=0.0)) * leg_cost(te, beta, costs)).sum()
        row.update(gross=gross/E, cost=cost/E, net=(gross-cost)/E,
                   ops=int(((pos[1:] != 0) & (pos[:-1] == 0)).sum() + (pos[0] != 0)), k=k, e=e)
        rows.append(row)
    return pd.DataFrame(rows)

def summarize(res, label):
    g, n = res.gross, res.net
    be = g.mean() / res.cost.mean() if res.cost.mean() > 0 else np.nan
    return pd.Series({'bruto': g.mean(), 'p_bruto': stats.ttest_1samp(g, 0).pvalue,
        'neto': n.mean(), 'p_neto': stats.ttest_1samp(n, 0).pvalue, 'neto>0': f"{(n>0).sum()}/{len(n)}",
        'abstenidas': int((res.ops == 0).sum()), 'ops/vent': res.ops.mean(), 'break-even (x costo)': be}, name=label)

In [19]:
datasets = {'1h': data_1h, '4h': data_4h, '8h': data_8h}
rows = []
for f, d in datasets.items():
    for td in (120, 240):
        for m in ('fixed', 'ou'):
            rows.append(summarize(run_cfg(d, td, m), f'{f}_{m}_{td}'))
tabla_freq = pd.DataFrame(rows)
display(tabla_freq.round(5))

,bruto,p_bruto,neto,p_neto,neto>0,abstenidas,ops/vent,break-even (x costo)
1h_fixed_120,-0.00001,0.98546,-0.00024,0.69375,8/19,4,1.26316,-0.04804
1h_ou_120,0.00017,0.90339,-0.00034,0.80088,11/19,0,2.73684,0.33206
1h_fixed_240,0.00016,0.85779,-0.00003,0.97559,8/19,5,1.05263,0.85475
1h_ou_240,-0.00096,0.45932,-0.00127,0.32173,10/19,0,1.94737,-3.03657
4h_fixed_120,0.00047,0.58098,0.00025,0.76849,10/19,3,1.15789,2.15736
4h_ou_120,-0.00020,0.89433,-0.00059,0.69068,11/19,0,2.10526,-0.50812
4h_fixed_240,0.00001,0.99107,-0.00015,0.88042,6/19,6,0.89474,0.06938
4h_ou_240,-0.00113,0.39674,-0.00138,0.29935,10/19,0,1.63158,-4.45813
8h_fixed_120,-0.00047,0.61541,-0.00065,0.49387,7/19,4,1.00000,-2.62060
8h_ou_120,-0.00006,0.96577,-0.00039,0.76864,11/19,0,1.84211,-0.17254


## Fase 4 – Costos reales por pata
Reemplazar el 1 pb uniforme por el costo medio de cada divisa (spread en pips convertido a pb). Recalcular neto y break-even. Pregunta: ¿era realista el colchón de ~1 pb?

In [22]:
rows = []
for mult in (1, 2, 3, 5):
    costs = {k: v*mult for k, v in COST_BPS.items()}
    for f, d in datasets.items():
        rows.append(summarize(run_cfg(d, 120, 'fixed', costs=costs), f'{f}_fixed_120_x{mult}'))
display(pd.DataFrame(rows)[['bruto','neto','p_neto','ops/vent','break-even (x costo)']].round(5))

,bruto,neto,p_neto,ops/vent,break-even (x costo)
1h_fixed_120_x1,-0.00001,-0.00024,0.69375,1.26316,-0.04804
4h_fixed_120_x1,0.00047,0.00025,0.76849,1.15789,2.15736
8h_fixed_120_x1,-0.00047,-0.00065,0.49387,1.00000,-2.62060
1h_fixed_120_x2,-0.00001,-0.00047,0.45011,1.26316,-0.02402
4h_fixed_120_x2,0.00047,0.00003,0.96839,1.15789,1.07868
8h_fixed_120_x2,-0.00047,-0.00083,0.39017,1.00000,-1.31030
1h_fixed_120_x3,-0.00001,-0.00071,0.27398,1.26316,-0.01601
4h_fixed_120_x3,0.00047,-0.00018,0.83422,1.15789,0.71912
8h_fixed_120_x3,-0.00047,-0.00101,0.30428,1.00000,-0.87353
1h_fixed_120_x5,-0.00001,-0.00117,0.09230,1.26316,-0.00961


## Fase 5 – Placebo
Repetir el pipeline sobre caminatas aleatorias en cada frecuencia (más simulaciones que antes) para confirmar que el neto esperado sin señal es ≤ 0, y cerrar con la decisión sobre esta canasta.

In [25]:
def placebo_freq(n_sims=20, seed=123):
    rng = np.random.default_rng(seed)
    sd = np.log(data_1h).diff().dropna().std().values
    out = []
    for i in range(n_sims):
        steps = rng.standard_normal(data_1h.shape) * sd
        fake1 = pd.DataFrame(data_1h.iloc[0].values * np.exp(np.cumsum(steps, axis=0)),
                             index=data_1h.index, columns=data_1h.columns)
        fakes = {'1h': fake1, '4h': resample_bars(fake1, '4h'), '8h': resample_bars(fake1, '8h')}
        for f, d in fakes.items():
            for td in (120, 240):
                for m in ('fixed', 'ou'):
                    r = run_cfg(d, td, m, seed=i)
                    out.append(dict(sim=i, cfg=f'{f}_{m}_{td}', neto=r.net.mean()))
    return pd.DataFrame(out)

pl = placebo_freq()
por_sim = pl.groupby('sim').agg(max_neto=('neto', 'max'), n_pos=('neto', lambda x: (x > 0).sum()))
mejor_real = tabla_freq['neto'].max()
print('Configs positivas por simulación (de 12): media', por_sim.n_pos.mean().round(1))
print('Mejor neto real:', round(mejor_real, 5))
print('Fracción de simulaciones donde la mejor config iguala o supera a la real:', (por_sim.max_neto >= mejor_real).mean())

Configs positivas por simulación (de 12): media 6.4
Mejor neto real: 0.00025
Fracción de simulaciones donde la mejor config iguala o supera a la real: 0.9


## Conclusión — Remuestreo y Costos

### **Pregunta**
¿Cambia el resultado de la canasta USDCAD/USDNOK/USDSEK/USDSGD al remuestrear a 4h u 8h y al variar costos?

### **Resultado**

No. Con contabilidad corregida (pesos congelados, costo por pata), el bruto normalizado es ≈ 0 en las 12 configuraciones (frecuencia × calibración × regla de umbrales): entre −0.0012 y +0.0005, con p_bruto de 0.37 a 0.99. El neto va de −0.0015 a +0.00025, con p_neto de 0.29 a 0.98. La sensibilidad a costos (×1 a ×5) solo empeora el neto, como corresponde a un bruto nulo. Remuestrear a 4h u 8h no recupera señal.

### **La única configuración con neto positivo**

(`4h_fixed_120`, +0.00025, p = 0.77) No es un hallazgo. Su neto desaparece con costo ×2 (+0.00003, p = 0.97) y se vuelve negativo con ×3. Sus vecinas (`4h_ou_120`, `4h_fixed_240`) son negativas o nulas.

### **Placebo (12 configuraciones sobre caminatas aleatorias, 20 simulaciones)** 

En promedio 6.4 de las 12 configuraciones salen positivas por azar, y en el 90% de las simulaciones la mejor configuración iguala o supera a la mejor real. El resultado más favorable de la tabla es indistinguible de lo que produce el azar al elegir la mejor de doce.

### **Diagnóstico**

Las operaciones por ventana son muy pocas (≈ 1 a 2.7; entre 0 y 6 ventanas de 19 sin ninguna), así que cada configuración se evalúa con ~20 operaciones en total. Probable causa (no verificada): al anclar el spread a la media del entrenamiento, el z se aleja de la banda cuando la relación se desplaza y el stop y el bloqueo impiden re-entrar. Con tan pocas operaciones, la potencia estadística es baja y ampliar las ventanas de prueba no la aumenta, porque el total de operaciones es el mismo.

### **Limitaciones**

Costos supuestos (la Fase 4 con spreads reales por par no cambia la conclusión porque el bruto es nulo); una sola canasta elegida con los mismos datos; ~19 ventanas; las pruebas t no corrigen por múltiples configuraciones.

### **Decisión**

No hay evidencia de un borde rentable en esta canasta a frecuencias de 1h, 4h u 8h. La mejora posible no está en el ajuste de umbrales, frecuencia o costos, sino en la **amplitud**: más pares o canastas para tener más operaciones independientes, evaluadas con placebo corregido por múltiples comparaciones. Siguiente notebook: amplitud y combinación de señales débiles, con el costo por par como criterio de selección.